In [ ]:
import sys
import os
from pathlib import Path


def _add_if_repo_root(path: Path) -> bool:
    path = path.resolve()
    if (path / "src" / "ml" / "retrain_gate.py").exists() and (path / "src" / "ml" / "__init__.py").exists():
        if str(path) not in sys.path:
            sys.path.insert(0, str(path))
        return True
    return False


# 1) Current working directory and parents.
# On newer Databricks runtimes, the notebook/script directory is commonly the CWD.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if _add_if_repo_root(candidate):
        print(f"OK: added repo root from cwd search: {candidate}")
        break
else:
    # 2) Resolve from notebook workspace path.
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    notebook_path = ctx.notebookPath().get()

    # notebook_path is usually like:
    # /Users/<user>/.bundle/healthcare-claim-ops/dev/files/src/notebooks/check_new_data
    workspace_notebook_path = Path("/Workspace") / notebook_path.lstrip("/")

    found = False
    for candidate in [workspace_notebook_path, *workspace_notebook_path.parents]:
        if _add_if_repo_root(candidate):
            print(
                f"OK: added repo root from notebook path search: {candidate}")
            found = True
            break

    if not found:
        print("DEBUG cwd:", Path.cwd())
        print("DEBUG notebook_path:", notebook_path)
        print("DEBUG workspace_notebook_path:", workspace_notebook_path)
        print("DEBUG first sys.path entries:", sys.path[:10])
        raise RuntimeError(
            "Could not locate repo root containing src/ml/retrain_gate.py")


print("OK: imported src.ml.retrain_gate and FEATURE_COLUMNS")

In [ ]:
from src.ml import FEATURE_COLUMNS
from src.ml.retrain_gate import decide_retrain

dbutils.widgets.text("catalog", "healthcare", "Catalog")
dbutils.widgets.text("gold_schema", "gold", "Gold schema")
dbutils.widgets.text("ml_schema", "ml", "ML schema")
dbutils.widgets.text("registered_model_name",
                     "healthcare.ml.claim_denial_model", "Registered model name")
dbutils.widgets.text("champion_alias", "champion", "Champion alias")

catalog = dbutils.widgets.get("catalog").strip()
gold_schema = dbutils.widgets.get("gold_schema").strip()
ml_schema = dbutils.widgets.get("ml_schema").strip()
registered_model_name = dbutils.widgets.get("registered_model_name").strip()
champion_alias = dbutils.widgets.get("champion_alias").strip()
gold_table = f"{catalog}.{gold_schema}.claim_features"

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
)

# 1. Compute retrain decision FIRST.
decision = decide_retrain(
    spark,
    gold_table=gold_table,
    feature_columns=list(FEATURE_COLUMNS),
    registered_model_name=registered_model_name,
    champion_alias=champion_alias,
)

print(decision.summary_line())

# 2. Explicit schema avoids Spark failing on None values like champion_run_id.
decision_schema = StructType(
    [
        StructField("should_retrain", StringType(), False),
        StructField("reason", StringType(), False),
        StructField("current_row_count", LongType(), False),
        StructField("current_gold_version", LongType(), False),
        StructField("current_fingerprint", StringType(), False),
        StructField("champion_run_id", StringType(), True),
    ]
)

decision_df = spark.createDataFrame(
    [
        (
            str(decision.should_retrain).lower(),
            str(decision.reason),
            int(decision.current_row_count),
            int(decision.current_gold_version),
            str(decision.current_fingerprint),
            None if decision.champion_run_id is None else str(
                decision.champion_run_id),
        )
    ],
    schema=decision_schema,
)

decision_df.createOrReplaceTempView("_current_retrain_decision")

spark.sql(f"""
INSERT INTO {catalog}.{ml_schema}.retrain_decisions
SELECT
  current_timestamp() AS decided_at,
  should_retrain,
  reason,
  current_row_count,
  current_gold_version,
  current_fingerprint,
  champion_run_id
FROM _current_retrain_decision
""")

# 3. Set task values for downstream condition task.
dbutils.jobs.taskValues.set(
    key="should_retrain",
    value=str(decision.should_retrain).lower(),
)
dbutils.jobs.taskValues.set(
    key="reason",
    value=decision.reason,
)
dbutils.jobs.taskValues.set(
    key="current_training_row_count",
    value=str(decision.current_row_count),
)
dbutils.jobs.taskValues.set(
    key="current_gold_version",
    value=str(decision.current_gold_version),
)
dbutils.jobs.taskValues.set(
    key="current_data_fingerprint",
    value=decision.current_fingerprint,
)

print("OK: retrain decision written and task values set")